# Export YOLO checkpoint to ONNX
Upload `mask_model.pt` when prompted. PyTorch is used only for this conversion step in Colab. Flask inference runs with ONNX Runtime.

In [ ]:
!pip -q install ultralytics onnx onnxsim

In [ ]:
from google.colab import files
uploaded = files.upload()
checkpoint = next(name for name in uploaded if name.lower().endswith('.pt'))
print('Checkpoint:', checkpoint)

In [ ]:
from ultralytics import YOLO
model = YOLO(checkpoint)
onnx_path = model.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False, batch=1)
print('Exported:', onnx_path)

In [ ]:
files.download(onnx_path)

## Load deployment settings from your local `.env`
Run this cell in Colab and upload your `.env` file when prompted. It loads the settings into the runtime without printing their values.

In [ ]:
from google.colab import files
from dotenv import dotenv_values
from io import StringIO
import os

env_upload = files.upload()
env_filename = next(name for name in env_upload if name.lower() == '.env')
env_values = dotenv_values(stream=StringIO(env_upload[env_filename].decode('utf-8')))
allowed_keys = {
    'HTML_TEMPLATE', 'HOST', 'PORT', 'AWS_ACCESS_KEY_ID',
    'AWS_SECRET_ACCESS_KEY', 'AWS_REGION', 'AWS_S3_BUCKET_NAME',
    'DIRECT_URL', 'DATABASE_URL', 'TELEGRAM_BOT_TOKEN', 'TELEGRAM_CHAT_ID'
}
for key, value in env_values.items():
    if key in allowed_keys and value is not None:
        os.environ[key] = value
os.environ.setdefault('MODEL_PATH', os.path.abspath('mask_model.onnx'))
print('Đã nạp các biến môi trường:', ', '.join(sorted(k for k in env_values if k in allowed_keys and env_values[k] is not None)))